## 0. 설치 검증

In [22]:
import shutil
import sys

print("Python:", sys.version.split()[0])
print("가상환경 경로:", sys.prefix)
print("가상환경 안에서 실행 중?", ".venv" in sys.prefix)
print()

# OS 패키지 확인 (pip로 설치되지 않는 것들)
print("Poppler  :", "OK" if shutil.which("pdftoppm") else "✗ 미설치 — Path 확인 후 재시작")
print("Tesseract:", "OK" if shutil.which("tesseract") else "✗ 미설치 — Path 확인 후 재시작")
print()

# 파이썬 패키지 확인
for mod in ["langchain", "unstructured", "chromadb", "tiktoken", "PIL"]:
    try:
        __import__(mod)
        print(f"{mod:14s}: OK")
    except ImportError:
        print(f"{mod:14s}: ✗ 미설치")

Python: 3.11.16
가상환경 경로: /Users/mac/Desktop/HyundaiAI/multimodal-rag/.venv
가상환경 안에서 실행 중? True

Poppler  : OK
Tesseract: OK

langchain     : OK
unstructured  : OK
chromadb      : OK
tiktoken      : OK
PIL           : OK


## 1. 데이터 추출 및 분할 — src/extract.py

In [23]:
import os
import sys

# 현재 작업 디렉터리 경로를 반환합니다.
current_directory = os.getcwd()
sys.path.append(os.path.dirname(current_directory))   # src 를 불러오기 위해 프로젝트 루트를 경로에 추가

from src.extract import extract_pdf_elements, categorize_elements

# fname: 처리할 PDF 파일 이름 / fpath: PDF가 저장된 폴더 경로
fname = "crop_area_2026_autumn.pdf"
fpath = os.path.join(os.path.dirname(current_directory), "data/")

print("현재 스크립트의 위치:", current_directory)
print("pdf 위치:", fpath)

# 실행하면 2~3분 걸립니다. 멈춘 게 아니니 기다리세요.
raw_pdf_elements = extract_pdf_elements(fpath, fname)
texts, tables = categorize_elements(raw_pdf_elements)

No languages specified, defaulting to English.


현재 스크립트의 위치: /Users/mac/Desktop/HyundaiAI/multimodal-rag/notebooks
pdf 위치: /Users/mac/Desktop/HyundaiAI/multimodal-rag/data/


In [24]:
from langchain_text_splitters import CharacterTextSplitter

# chunk_size=2000: 각 텍스트 블록의 최대 크기
# chunk_overlap=200: 청크 간 200자의 겹침을 허용해 중요한 정보를 잃지 않도록 함
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=2000, chunk_overlap=200
)

# 텍스트 리스트를 하나의 긴 문자열로 결합한 뒤 2000자 단위로 분할
joined_texts = " ".join(texts)
texts_2k_token = text_splitter.split_text(joined_texts)

print("분할 후 블록 수:", len(texts_2k_token))
print("원본 블록 수:", len(texts))
print(texts[0])
print(type(texts_2k_token))

분할 후 블록 수: 3
원본 블록 수: 11
농업면적조사

©) 2 teleler

보도자료

coy

보도시점 2026. 10. 30.(금) 12:00 배포 2026. 10. 30.(금) 08:30

2026년 재배면적조사 결과 - 가을배추, 가을무 -
<class 'list'>


In [25]:
figures_path = os.path.join(current_directory, "figures/")
image_files = sorted(f for f in os.listdir(figures_path) if f.endswith((".jpg", ".png", ".jpeg")))
print("figures 위치:", figures_path)
print("추출된 이미지 수:", len(image_files))
print("표 수:", len(tables))
print(image_files[:5])

figures 위치: /Users/mac/Desktop/HyundaiAI/multimodal-rag/notebooks/figures/
추출된 이미지 수: 22
표 수: 10
['figure-1-1.jpg', 'figure-1-10.jpg', 'figure-1-11.jpg', 'figure-1-12.jpg', 'figure-1-2.jpg']


## 2. 텍스트·표 요약 — src/summarize.py

In [26]:
import os
from dotenv import load_dotenv

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OpenAI API 키가 없습니다. 한번 더 확인 부탁드립니다.")

In [27]:
import pickle
from src.summarize import generate_text_summaries

# 요약은 실행할 때마다 조금씩 달라져 검색 결과도 달라집니다.
# 결과를 재현할 수 있도록, 지금 추출 결과와 똑같은 입력으로 만든 요약이 저장돼 있으면 불러오고 없으면 새로 만들어 저장합니다.
summaries_path = os.path.join(os.path.dirname(current_directory), "data", "processed", "summaries.pkl")
cache = None
if os.path.exists(summaries_path):
    with open(summaries_path, "rb") as f:
        cache = pickle.load(f)
    if cache["texts_2k_token"] != texts_2k_token or cache["tables"] != tables or cache["image_files"] != image_files:
        cache = None   # 추출 결과가 달라졌으면 저장본을 쓰지 않음

if cache:
    text_summaries, table_summaries = cache["text_summaries"], cache["table_summaries"]
    print("저장된 요약을 불러왔습니다:", summaries_path)
else:
    text_summaries, table_summaries = generate_text_summaries(
        texts_2k_token, tables, summarize_texts=True
    )
print(text_summaries)
print(table_summaries)

저장된 요약을 불러왔습니다: /Users/mac/Desktop/HyundaiAI/multimodal-rag/data/processed/summaries.pkl
['**2026년 가을배추·무 재배면적조사 요약**\n\n- **조사 기간**: 2026년 9월 14일부터 10월 12일까지\n- **재배면적 결과**: \n  - 가을배추: 13,909ha (전년 대비 5.5% 증가, 13,182ha)\n  - 가을무: 결과는 추후 공표 예정\n- **통계 특성**: 표본조사로 인한 표본오차 및 비표본오차 포함\n- **면적 단위**: 1ha = 10,000㎡\n- **변경 사항**: \n  - 2024년부터 군위군은 대구광역시로 집계\n  - 2026년 7월부터 전남광주통합특별시 추가\n- **자료 이용**: 국가데이터처 및 국가통계포털에서 확인 가능\n- **문의처**: 국가데이터처 농어업통계과 (042-481-6954)', '**2026년 가을배추·무 재배면적 조사 요약**\n\n- **가을배추 재배면적**: 13,909ha (전년 대비 5.5% 증가)\n- **가을무 재배면적**: 5,749ha (전년 대비 1.3% 감소)\n\n**세부 사항**:\n1. **가을배추**:\n   - 주요 재배 지역: 전남광주(3,686ha), 충북(2,130ha), 강원(1,779ha), 전북(1,629ha), 경북(1,613ha)\n   - 연도별 추세: 2016년 11,428ha에서 2026년 13,909ha로 증가\n\n2. **가을무**:\n   - 주요 재배 지역: 경기(1,208ha), 전북(1,082ha), 충남(1,010ha), 전남광주(772ha), 강원(662ha)\n   - 연도별 추세: 2016년 6,340ha에서 2026년 5,749ha로 감소\n\n**통계 요약**:\n- 2025년 가을배추: 13,182ha, 가을무: 5,826ha\n- 2026년 가을배추: 13,909ha, 가을무: 5,749ha\n\n이 데이터는 가을배추와 가을무의 재배면적 변화를

## 3. 이미지 요약 — src/summarize.py

In [28]:
from src.summarize import encode_image, generate_img_summaries

# "그림이 문자열이 되었다" 확인 — 앞 100자만 출력
print(encode_image(os.path.join(figures_path, image_files[0]))[:100])

/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAx


In [29]:
if cache:
    img_base64_list, image_summaries = cache["img_base64_list"], cache["image_summaries"]
else:
    img_base64_list, image_summaries = generate_img_summaries(figures_path)
    os.makedirs(os.path.dirname(summaries_path), exist_ok=True)
    with open(summaries_path, "wb") as f:
        pickle.dump({
            "texts_2k_token": texts_2k_token, "tables": tables, "image_files": image_files,
            "text_summaries": text_summaries, "table_summaries": table_summaries,
            "img_base64_list": img_base64_list, "image_summaries": image_summaries,
        }, f)
    print("요약을 저장했습니다:", summaries_path)
print(image_summaries[0])

국가통계 로고, 청색과 녹색 색상 사용, 통계 그래프 형상 포함.


## 4. 다중 벡터 검색기와 최종 체인 — src/retriever.py

In [30]:
from src.retriever import retriever, add_documents, split_image_text_types, chain

if text_summaries:
    add_documents(retriever, text_summaries, texts_2k_token)
if table_summaries:
    add_documents(retriever, table_summaries, tables)
if image_summaries:
    add_documents(retriever, image_summaries, img_base64_list)

print("벡터 DB에 저장된 요약 수:", len(retriever.vectorstore.get()["ids"]))

벡터 DB에 저장된 요약 수: 70


In [31]:
query = "2006년 가을배추와 가을무 재배면적은 각각 몇 ha야?"

found = split_image_text_types(retriever.invoke(query))
print("검색된 이미지:", len(found["images"]), "건 / 텍스트·표:", len(found["texts"]), "건")
print(chain.invoke(query))

검색된 이미지: 2 건 / 텍스트·표: 2 건
2006년 가을배추 재배면적은 14,368ha, 가을무 재배면적은 9,758ha입니다.


## 5. Before / After 

In [32]:
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

text_vectorstore = Chroma(collection_name="text_only_rag", embedding_function=OpenAIEmbeddings())
text_vectorstore.add_texts(texts_2k_token)
text_retriever = text_vectorstore.as_retriever(search_kwargs={"k": 4})

text_prompt = ChatPromptTemplate.from_template(
    "아래 자료를 근거로 질문에 한국어로 답하세요.\n"
    "질문에 나온 기준 시점은 답변에 그대로 포함하세요.\n"
    "자료에 없는 내용은 모른다고 답하세요.\n\n"
    "질문: {question}\n\n"
    "참고 텍스트:\n{context}"
)

text_only_chain = (
    {
        "context": text_retriever | RunnableLambda(lambda docs: "\n".join(d.page_content for d in docs)),
        "question": RunnablePassthrough(),
    }
    | text_prompt
    | ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=1024)
    | StrOutputParser()
)

print("=" * 60)
print("질문:", query)
print("=" * 60)
print("\n[BEFORE] 텍스트 전용 RAG")
print("-" * 60)
print(text_only_chain.invoke(query))
print("\n[AFTER] 멀티모달 RAG")
print("-" * 60)
print(chain.invoke(query))

질문: 2006년 가을배추와 가을무 재배면적은 각각 몇 ha야?

[BEFORE] 텍스트 전용 RAG
------------------------------------------------------------
2006년 가을배추와 가을무 재배면적에 대한 정보는 제공된 자료에 없습니다. 따라서 2006년 가을배추와 가을무 재배면적은 알 수 없습니다.

[AFTER] 멀티모달 RAG
------------------------------------------------------------
2006년 가을배추 재배면적은 14,368ha, 가을무 재배면적은 9,758ha입니다.


## 6. 질의응답 5건

| # | 정답 출처 | 정답 |
| --- | --- | --- |
| 1 | 5쪽 연도별 가을배추·무 재배면적 추이 그래프, 10쪽 통계표 | 가을배추 14,368ha, 가을무 9,758ha |
| 2 | 5쪽 결과 요약 표 | 13,909ha, 전년 대비 +5.5% |
| 3 | 5쪽 결과 요약 표 | 5,749ha, 전년 대비 -1.3% |
| 4 | 7쪽 상위 8개 시도별 가을배추 그래프·표 | 전남광주 3,686ha, 충북 2,130ha, 강원 1,779ha |
| 5 | 9쪽 상위 8개 시도별 가을무 그래프·표 | 경기 1,208ha |

In [33]:
questions = [
    "2006년 가을배추와 가을무 재배면적은 각각 몇 ha야?",
    "2026년 가을배추 재배면적은 몇 ha이고 전년 대비 증감률은 몇 %야?",
    "2026년 가을무 재배면적은 몇 ha이고 전년 대비 증감률은 몇 %야?",
    "2026년 가을배추 재배면적 상위 3개 시도와 각각의 면적은?",
    "2026년 가을무 재배면적이 가장 넓은 시도와 그 면적은?",
]

print("| # | 질문 | 답변 | 근거 원본 종류 |")
print("| --- | --- | --- | --- |")
for i, q in enumerate(questions, 1):
    ctx = split_image_text_types(retriever.invoke(q))
    kinds = []
    if ctx["images"]:
        kinds.append("이미지")
    if any(t in tables for t in ctx["texts"]):
        kinds.append("표")
    if any(t not in tables for t in ctx["texts"]):
        kinds.append("텍스트")
    ans = chain.invoke(q).replace("\n", " ")
    print(f"| {i} | {q} | {ans} | {', '.join(kinds)} |")

| # | 질문 | 답변 | 근거 원본 종류 |
| --- | --- | --- | --- |
| 1 | 2006년 가을배추와 가을무 재배면적은 각각 몇 ha야? | 2006년 가을배추 재배면적은 14,368ha, 가을무 재배면적은 9,758ha입니다. | 이미지, 텍스트 |
| 2 | 2026년 가을배추 재배면적은 몇 ha이고 전년 대비 증감률은 몇 %야? | 자료에 따르면 2026년 가을배추 재배면적은 2ha이고, 전년 대비 증감률은 0%입니다. | 이미지 |
| 3 | 2026년 가을무 재배면적은 몇 ha이고 전년 대비 증감률은 몇 %야? | 자료에 따르면 2026년 가을무 재배면적은 2,000ha이고, 전년 대비 증감률은 5%입니다. | 이미지 |
| 4 | 2026년 가을배추 재배면적 상위 3개 시도와 각각의 면적은? | 자료에 따르면, 2026년 가을배추 재배면적 상위 3개 시도와 각각의 면적은 다음과 같습니다:  1. **전라남도**: 1,500ha 2. **경상남도**: 1,200ha 3. **충청남도**: 1,000ha  이 정보는 2026년 기준입니다. | 이미지 |
| 5 | 2026년 가을무 재배면적이 가장 넓은 시도와 그 면적은? | 자료에 없는 내용은 모른다고 답하세요. | 이미지 |
